# NetworkD3 Visualization of Macroeconomic Indicator Correlations

This notebook demonstrates how to create interactive network visualizations of synthetic correlations between macroeconomic indicators using Python and networkd3-style visualizations.

## Overview
We'll create a force-directed graph showing relationships between various macroeconomic indicators such as:
- GDP Growth
- Inflation Rate
- Unemployment Rate
- Interest Rates
- Stock Market Index
- Currency Exchange Rate
- Trade Balance
- Consumer Confidence

In [ ]:
# Import required libraries
import numpy as np
import pandas as pd
import networkx as nx
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import random
from itertools import combinations
import matplotlib.pyplot as plt
import seaborn as sns

# Set random seed for reproducibility
np.random.seed(42)
random.seed(42)

## 1. Generate Synthetic Macroeconomic Indicators Data

In [ ]:
# Define macroeconomic indicators
indicators = [
    "GDP Growth",
    "Inflation Rate", 
    "Unemployment Rate",
    "Federal Funds Rate",
    "10-Year Treasury Yield",
    "S&P 500 Index", 
    "USD/EUR Exchange Rate",
    "Trade Balance",
    "Consumer Confidence Index",
    "Housing Price Index",
    "Oil Price (WTI)",
    "Gold Price",
    "VIX (Volatility Index)",
    "Industrial Production",
    "Retail Sales Growth"
]

n_indicators = len(indicators)
print(f"Number of indicators: {n_indicators}")
print("Indicators:", indicators)

In [ ]:
# Generate synthetic correlation matrix with realistic economic relationships
def generate_economic_correlations(indicators):
    n = len(indicators)
    
    # Start with identity matrix
    corr_matrix = np.eye(n)
    
    # Define some realistic economic relationships
    economic_relationships = {
        ("GDP Growth", "Unemployment Rate"): -0.7,  # Strong negative correlation
        ("GDP Growth", "S&P 500 Index"): 0.6,       # Positive correlation
        ("GDP Growth", "Consumer Confidence Index"): 0.8,  # Strong positive
        ("Inflation Rate", "Federal Funds Rate"): 0.7,     # Central bank response
        ("Inflation Rate", "10-Year Treasury Yield"): 0.6, # Bond yields rise with inflation
        ("Unemployment Rate", "Consumer Confidence Index"): -0.6,  # Negative relationship
        ("Federal Funds Rate", "10-Year Treasury Yield"): 0.8,     # Yield curve relationship
        ("S&P 500 Index", "VIX (Volatility Index)"): -0.8,         # Fear vs greed
        ("S&P 500 Index", "Consumer Confidence Index"): 0.7,       # Market confidence
        ("Oil Price (WTI)", "Inflation Rate"): 0.5,                # Energy inflation
        ("Gold Price", "VIX (Volatility Index)"): 0.4,             # Safe haven demand
        ("Housing Price Index", "GDP Growth"): 0.6,                # Economic growth
        ("Industrial Production", "GDP Growth"): 0.9,              # Very strong relationship
        ("Retail Sales Growth", "Consumer Confidence Index"): 0.7, # Consumer spending
        ("Trade Balance", "USD/EUR Exchange Rate"): -0.3,          # Currency strength
    }
    
    # Apply predefined relationships
    for (ind1, ind2), corr in economic_relationships.items():
        if ind1 in indicators and ind2 in indicators:
            i = indicators.index(ind1)
            j = indicators.index(ind2)
            corr_matrix[i, j] = corr
            corr_matrix[j, i] = corr  # Symmetric matrix
    
    # Fill remaining correlations with small random values
    for i in range(n):
        for j in range(i+1, n):
            if corr_matrix[i, j] == 0:  # Not yet filled
                # Generate small random correlation
                random_corr = np.random.uniform(-0.3, 0.3)
                corr_matrix[i, j] = random_corr
                corr_matrix[j, i] = random_corr
    
    return corr_matrix

# Generate the correlation matrix
correlation_matrix = generate_economic_correlations(indicators)

# Create DataFrame for easier handling
corr_df = pd.DataFrame(correlation_matrix, 
                      index=indicators, 
                      columns=indicators)

print("Correlation matrix shape:", corr_df.shape)
print("\nSample correlations:")
print(corr_df.iloc[:5, :5].round(3))

## 2. Create Network Graph Data Structure

In [ ]:
# Create network graph from correlation matrix
def create_network_from_correlations(corr_df, threshold=0.3):
    """
    Create network graph from correlation matrix.
    Only include edges where |correlation| > threshold.
    """
    G = nx.Graph()
    
    # Add nodes (indicators)
    for indicator in corr_df.index:
        G.add_node(indicator)
    
    # Add edges (correlations above threshold)
    for i, indicator1 in enumerate(corr_df.index):
        for j, indicator2 in enumerate(corr_df.columns):
            if i < j:  # Avoid duplicate edges
                correlation = corr_df.iloc[i, j]
                if abs(correlation) > threshold:
                    G.add_edge(indicator1, indicator2, 
                              weight=abs(correlation),
                              correlation=correlation,
                              edge_type='positive' if correlation > 0 else 'negative')
    
    return G

# Create the network
network = create_network_from_correlations(corr_df, threshold=0.3)

print(f"Network created with {network.number_of_nodes()} nodes and {network.number_of_edges()} edges")
print(f"Network density: {nx.density(network):.3f}")

## 3. Interactive NetworkD3-Style Visualization with Plotly

In [ ]:
# Create interactive network visualization
def create_interactive_network_plot(G, title="Macroeconomic Indicators Network"):
    """
    Create an interactive network plot using Plotly that mimics networkd3 style.
    """
    # Get node positions using spring layout
    pos = nx.spring_layout(G, k=3, iterations=50, seed=42)
    
    # Prepare edge traces
    edge_x = []
    edge_y = []
    edge_colors = []
    edge_widths = []
    edge_info = []
    
    for edge in G.edges(data=True):
        x0, y0 = pos[edge[0]]
        x1, y1 = pos[edge[1]]
        
        edge_x.extend([x0, x1, None])
        edge_y.extend([y0, y1, None])
        
        correlation = edge[2]['correlation']
        weight = edge[2]['weight']
        
        # Color based on correlation type
        color = 'rgba(255, 0, 0, 0.6)' if correlation < 0 else 'rgba(0, 100, 255, 0.6)'
        edge_colors.extend([color, color, color])
        
        # Width based on correlation strength
        width = max(1, weight * 5)
        edge_widths.extend([width, width, width])
        
        edge_info.append(f"{edge[0]} ↔ {edge[1]}: {correlation:.3f}")
    
    # Create edge trace
    edge_trace = go.Scatter(
        x=edge_x, y=edge_y,
        line=dict(width=2, color='rgba(125, 125, 125, 0.5)'),
        hoverinfo='none',
        mode='lines'
    )
    
    # Prepare node traces
    node_x = []
    node_y = []
    node_text = []
    node_colors = []
    node_sizes = []
    
    # Define categories for coloring nodes
    node_categories = {
        'Economic Growth': ['GDP Growth', 'Industrial Production', 'Retail Sales Growth'],
        'Monetary Policy': ['Federal Funds Rate', '10-Year Treasury Yield', 'Inflation Rate'],
        'Labor Market': ['Unemployment Rate'],
        'Financial Markets': ['S&P 500 Index', 'VIX (Volatility Index)', 'Gold Price'],
        'Consumer Sentiment': ['Consumer Confidence Index'],
        'Real Estate': ['Housing Price Index'],
        'Commodities': ['Oil Price (WTI)'],
        'International': ['USD/EUR Exchange Rate', 'Trade Balance']
    }
    
    # Create reverse mapping
    indicator_to_category = {}
    for category, indicators_list in node_categories.items():
        for indicator in indicators_list:
            indicator_to_category[indicator] = category
    
    # Color palette for categories
    category_colors = {
        'Economic Growth': '#1f77b4',
        'Monetary Policy': '#ff7f0e', 
        'Labor Market': '#2ca02c',
        'Financial Markets': '#d62728',
        'Consumer Sentiment': '#9467bd',
        'Real Estate': '#8c564b',
        'Commodities': '#e377c2',
        'International': '#7f7f7f'
    }
    
    for node in G.nodes():
        x, y = pos[node]
        node_x.append(x)
        node_y.append(y)
        
        # Node info
        connections = list(G.neighbors(node))
        degree = G.degree(node)
        
        node_info = f"<b>{node}</b><br>"
        node_info += f"Connections: {degree}<br>"
        node_info += f"Category: {indicator_to_category.get(node, 'Other')}<br>"
        node_info += "Connected to:<br>" + "<br>".join([f"• {conn}" for conn in connections[:5]])
        if len(connections) > 5:
            node_info += f"<br>... and {len(connections)-5} more"
        
        node_text.append(node_info)
        
        # Color and size based on category and degree
        category = indicator_to_category.get(node, 'Other')
        node_colors.append(category_colors.get(category, '#17becf'))
        node_sizes.append(max(20, degree * 5))
    
    # Create node trace
    node_trace = go.Scatter(
        x=node_x, y=node_y,
        mode='markers+text',
        hoverinfo='text',
        hovertext=node_text,
        text=[node.replace(' ', '<br>') for node in G.nodes()],
        textposition="middle center",
        textfont=dict(size=10, color='white'),
        marker=dict(
            size=node_sizes,
            color=node_colors,
            line=dict(width=2, color='white'),
            opacity=0.8
        )
    )
    
    # Create the plot - FIXED: Remove titlefont_size and use proper title structure
    fig = go.Figure(data=[edge_trace, node_trace],
                   layout=go.Layout(
                        title=dict(
                            text=title,
                            x=0.5,
                            font=dict(size=20)  # Font size is set here within title dict
                        ),
                        # REMOVED: titlefont_size=16,  # This line causes the error
                        showlegend=False,
                        hovermode='closest',
                        margin=dict(b=20,l=5,r=5,t=40),
                        annotations=[ dict(
                            text="Hover over nodes for details. Node size = # of connections. Blue edges = positive correlation, Red edges = negative correlation",
                            showarrow=False,
                            xref="paper", yref="paper",
                            x=0.005, y=-0.002,
                            xanchor="left", yanchor="bottom",
                            font=dict(size=12)
                        )],
                        xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
                        yaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
                        plot_bgcolor='rgba(248,248,255,0.8)',
                        width=1000,
                        height=800
                   ))
    
    return fig

# Create and display the interactive plot
fig = create_interactive_network_plot(network)
fig.show()

## 4. Alternative Visualization: Correlation Heatmap

In [ ]:
# Create correlation heatmap
fig_heatmap = px.imshow(
    corr_df,
    color_continuous_scale='RdBu',
    color_continuous_midpoint=0,
    title="Macroeconomic Indicators Correlation Matrix",
    width=800,
    height=800
)

fig_heatmap.update_layout(
    title_x=0.5,
    xaxis_title="Indicators",
    yaxis_title="Indicators"
)

fig_heatmap.show()

## 5. Network Analysis and Statistics

In [ ]:
# Analyze network properties
def analyze_network(G):
    """
    Analyze various network properties.
    """
    analysis = {}
    
    # Basic properties
    analysis['num_nodes'] = G.number_of_nodes()
    analysis['num_edges'] = G.number_of_edges()
    analysis['density'] = nx.density(G)
    
    # Centrality measures
    analysis['degree_centrality'] = nx.degree_centrality(G)
    analysis['betweenness_centrality'] = nx.betweenness_centrality(G)
    analysis['closeness_centrality'] = nx.closeness_centrality(G)
    analysis['eigenvector_centrality'] = nx.eigenvector_centrality(G, max_iter=1000)
    
    # Clustering
    analysis['clustering_coefficient'] = nx.average_clustering(G)
    analysis['transitivity'] = nx.transitivity(G)
    
    # Path properties
    if nx.is_connected(G):
        analysis['avg_shortest_path'] = nx.average_shortest_path_length(G)
        analysis['diameter'] = nx.diameter(G)
    else:
        analysis['avg_shortest_path'] = 'Network not connected'
        analysis['diameter'] = 'Network not connected'
    
    return analysis

# Perform network analysis
network_stats = analyze_network(network)

print("=== Network Analysis ===")
print(f"Number of nodes: {network_stats['num_nodes']}")
print(f"Number of edges: {network_stats['num_edges']}")
print(f"Network density: {network_stats['density']:.3f}")
print(f"Average clustering coefficient: {network_stats['clustering_coefficient']:.3f}")
print(f"Transitivity: {network_stats['transitivity']:.3f}")
print(f"Average shortest path length: {network_stats['avg_shortest_path']}")
print(f"Network diameter: {network_stats['diameter']}")

print("\n=== Top 5 Most Central Indicators (by degree) ===")
degree_centrality_sorted = sorted(network_stats['degree_centrality'].items(), 
                                 key=lambda x: x[1], reverse=True)
for indicator, centrality in degree_centrality_sorted[:5]:
    print(f"{indicator}: {centrality:.3f}")

print("\n=== Top 5 Most Central Indicators (by betweenness) ===")
betweenness_centrality_sorted = sorted(network_stats['betweenness_centrality'].items(), 
                                      key=lambda x: x[1], reverse=True)
for indicator, centrality in betweenness_centrality_sorted[:5]:
    print(f"{indicator}: {centrality:.3f}")

## 6. Dynamic Network with Different Correlation Thresholds

In [ ]:
# Create networks with different correlation thresholds
thresholds = [0.2, 0.3, 0.4, 0.5, 0.6]

print("Network properties at different correlation thresholds:")
print("Threshold | Nodes | Edges | Density | Avg Clustering")
print("-" * 55)

for threshold in thresholds:
    G_thresh = create_network_from_correlations(corr_df, threshold=threshold)
    
    nodes = G_thresh.number_of_nodes()
    edges = G_thresh.number_of_edges()
    density = nx.density(G_thresh)
    clustering = nx.average_clustering(G_thresh)
    
    print(f"{threshold:8.1f} | {nodes:5d} | {edges:5d} | {density:7.3f} | {clustering:12.3f}")

## 7. Export Network Data for D3.js (Optional)

In [ ]:
import json

# Export network data in D3.js format
def export_network_for_d3(G, filename="network_data.json"):
    """
    Export network data in format suitable for D3.js force-directed graphs.
    """
    # Create nodes list
    nodes = []
    node_index = {node: i for i, node in enumerate(G.nodes())}
    
    for i, node in enumerate(G.nodes()):
        nodes.append({
            "id": node,
            "group": i % 8,  # Assign groups for coloring
            "degree": G.degree(node)
        })
    
    # Create links list
    links = []
    for edge in G.edges(data=True):
        links.append({
            "source": node_index[edge[0]],
            "target": node_index[edge[1]],
            "value": edge[2]['weight'],
            "correlation": edge[2]['correlation']
        })
    
    # Create the data structure
    data = {
        "nodes": nodes,
        "links": links
    }
    
    # Save to JSON file
    with open(filename, 'w') as f:
        json.dump(data, f, indent=2)
    
    print(f"Network data exported to {filename}")
    return data

# Export the network
d3_data = export_network_for_d3(network)
print(f"Exported {len(d3_data['nodes'])} nodes and {len(d3_data['links'])} edges")

## Summary

This notebook demonstrates how to create interactive network visualizations of macroeconomic indicator correlations using Python. The key features include:

1. **Synthetic Data Generation**: Created realistic correlation patterns between macroeconomic indicators
2. **Network Construction**: Built a graph where nodes are indicators and edges represent significant correlations
3. **Interactive Visualization**: Used Plotly to create networkd3-style force-directed graphs
4. **Network Analysis**: Calculated centrality measures and network statistics
5. **Multiple Views**: Provided both network graph and correlation heatmap visualizations
6. **Export Capability**: Option to export data for use with D3.js

The visualization helps identify:
- **Central indicators**: Those with many strong correlations (larger nodes)
- **Correlation types**: Positive (blue edges) vs negative (red edges) relationships  
- **Economic clusters**: Groups of closely related indicators
- **Network structure**: How macroeconomic indicators are interconnected

This approach can be extended with real economic data, time-varying correlations, or more sophisticated network analysis techniques.